In [1]:
from email import utils
import polars as pl
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.append("../src")  # Add path to where 'retail' package lives
import retail.retailfilter as rf # Correct import path
import utils.utils as ut  # Correct import path


[nltk_data] Downloading package names to /home/ogoodwin/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [2]:

# Input file
# input_file = Path("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
input_file = Path("../data/input_files/uk_foursquare_cc_nov_iceberg.parquet")


# Read in parquet
foursquare = pl.read_parquet(input_file)

#look for "Celebrate 365" in name
celebrate_365 = foursquare.filter(pl.col("name").str.contains("Celebrate 365")).to_pandas()
celebrate_365

,fsq_place_id,name,lat,lng,address,locality,region,postcode,admin_region,post_town,...,level2_category_id,level2_category_name,level3_category_id,level3_category_name,level4_category_id,level4_category_name,level5_category_id,level5_category_name,level6_category_id,level6_category_name
0,5adc0718b0b24ea5980d9f34,Celebrate 365,53.448605,-2.391028,8 Woodsend Rd,Urmston,Greater Manchester,M41 8QT,England,Manchester,...,None,None,None,None,None,None,None,None,None,None


In [ ]:

# Input file
# input_file = Path("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
input_file = Path("../data/input_files/uk_foursquare_cc_nov_iceberg.parquet")


# Read in parquet
foursquare = pl.read_parquet(input_file)

#look for "Celebrate 365" in name
celebrate_365 = foursquare.filter(pl.col("name").str.contains("Celebrate 365")).select(["fsq_place_id", "name", "fsq_category_labels"]).to_pandas()
print(len(foursquare))
# Filter
foursquare_filtnew = rf.filter_foursquare_invmethod(foursquare)
print(len(foursquare_filtnew))
celebrate_365 = foursquare_filtnew.filter(pl.col("name").str.contains("Celebrate 365")).to_pandas()


# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng", "address", "locality", "region",
    "postcode", "admin_region", "post_town", "po_box", "country",
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "fsq_category_labels", "h3_11"
]

# name has changed now its exploded.
if "fsq_category_ids" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_id" else "fsq_category_ids" for col in COLUMNS_TO_KEEP]
elif "fsq_category_id" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_ids" else "fsq_category_id" for col in COLUMNS_TO_KEEP]
else:
    COLUMNS_TO_KEEP_MOD = COLUMNS_TO_KEEP

# Add H3 column
foursquare_filtnew = ut.add_h3_11_column(foursquare_filtnew)

# Keep only selected columns
foursquare_filtnew = foursquare_filtnew.select(COLUMNS_TO_KEEP_MOD)

# Convert to pandas
fsq_df = foursquare_filtnew.to_pandas()

# Create geometry
fsq_df["geometry"] = [
    Point(xy) if pd.notnull(xy[0]) and pd.notnull(xy[1]) else None
    for xy in zip(fsq_df["lng"], fsq_df["lat"])
]
fsq_df = fsq_df[fsq_df["geometry"].notnull()]

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(fsq_df, geometry="geometry", crs="EPSG:4326")

# Generate output filename: "inputfilename_retailfiltnew_geo.parquet"
output_path = input_file.with_name(input_file.stem + "_retailfilt_geo.parquet")

# Save
gdf.to_parquet(output_path)
print(f"✅ Saved GeoDataFrame with H3 to {output_path}")

# Generate output filename: GeoPackage
output_gpkg = input_file.with_name(input_file.stem + "_retailfilt_geo.gpkg")
gdf.to_file(output_gpkg, driver="GPKG")
print(f"✅ Also saved GeoDataFrame as GeoPackage to {output_gpkg}")

4695350
Filtered Foursquare data (Retail Cats): 1868303 rows remaining.
Kept 1539  Argos 'Warehouse or Wholesale Store' entries after filtering.
Filtered Foursquare data (remove non-Argos warehouse): 1863742 rows remaining.
Filtered Foursquare data (find Costa Express): 1858432 rows remaining.
Filtered Foursquare data (remove post box): 1858380 rows remaining.
Filtered Foursquare data (remove closed): 1592000 rows remaining.
Filtered Foursquare data (remove cat dupes): 1427048 rows remaining.
1427048
✅ Saved GeoDataFrame with H3 to ../data/input_files/uk_foursquare_cc_nov_iceberg_retailfilt_geo.parquet
✅ Also saved GeoDataFrame as GeoPackage to ../data/input_files/uk_foursquare_cc_nov_iceberg_retailfilt_geo.gpkg


: 

In [15]:
cats = foursquare_filtnew["fsq_category_labels"].value_counts()
# Sort descending by count
cats = cats.sort("count", descending=True)
cats

#is ["Retail"] in there any more?
cats.filter(pl.col("fsq_category_labels") == ["Retail"])

fsq_category_labels,count
list[str],u32
